# 01-5 视频怎么变成 Token

在 [上一节](./01-4-image_to_token.ipynb) 中，我们了解了图片如何通过 ViT 的 Patch 机制变成视觉 Token。那视频呢？视频可以理解为"一组连续的图片"，但它的 Token 化方式有什么不同？

本节我们将探索 **视频 Token**：视频的时空信息如何被编码为 Token？

我们将继续使用 **Qwen3.5-0.8B** 模型，用一段开源的乖乖吉卜力风格视频来实际验证。

⏸ 关于音频的 Token 化，我们将在 [下一节](./01-6-audio_to_token.ipynb) 中深入探讨。

In [1]:
from modelscope import snapshot_download
from transformers import AutoTokenizer, AutoProcessor, Qwen3_5ForConditionalGeneration, TextIteratorStreamer
from PIL import Image
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import json
import os
import logging
from threading import Thread

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("modelscope").setLevel(logging.ERROR)

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'PingFang SC']
plt.rcParams['axes.unicode_minus'] = False

NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()

tokenizer_id = "Qwen/Qwen3.5-0.8B"
model_dir = snapshot_download(tokenizer_id)
tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
processor = AutoProcessor.from_pretrained(model_dir, local_files_only=True)
model = Qwen3_5ForConditionalGeneration.from_pretrained(model_dir, local_files_only=True)
config = model.config

print(f"模型: {tokenizer_id}")
print(f"词表大小: {len(tokenizer)}")
print(f"Processor 类型: {type(processor).__name__}")
print(f"模型加载完成!")

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

模型: Qwen/Qwen3.5-0.8B
词表大小: 248077
Processor 类型: Qwen3VLProcessor
模型加载完成!


## 1. 视频 Token：时间维度如何被编码？

视频可以理解为“一组连续的图片”。但如果每一帧都当作独立的图片来处理，Token 数量会爆炸（一段 10 秒的 30fps 视频就有 300 帧）。

Qwen3.5 的解决方案是引入 **时间维度的 Patch**：

1. **采样**：不是每一帧都处理，而是每秒采样 1-2 帧（即 1-2 FPS）。
2. **时间 Patch**：将连续的 `temporal_patch_size=2` 帧合并为一个时间 Patch。
3. **空间合并**：和图片一样，相邻的 $2 \times 2$ 个空间 Patch 合并。
4. **结果**：每个视频 Token 实际上编码了 $2 \times 2 \times 2 = 8$ 个原始 Patch 的信息（时间 2 x 空间 2x2）。

让我们用一段开源视频来实际验证：

In [2]:
temporal_patch_size = config.vision_config.temporal_patch_size
spatial_merge = config.vision_config.spatial_merge_size
patch_sz = config.vision_config.patch_size

print("=== 视频 Token 编码参数 ===")
print(f"时间 Patch 尺寸: {temporal_patch_size} (连续 {temporal_patch_size} 帧合并)")
print(f"空间合并尺寸: {spatial_merge}x{spatial_merge}")
print(f"Patch 尺寸: {patch_sz}x{patch_sz}")
print(f"每个视频 Token 实际编码: {temporal_patch_size} 帧 x {spatial_merge}x{spatial_merge} 空间 Patch = {temporal_patch_size * spatial_merge * spatial_merge} 个原始 Patch")

video_path = os.path.join(NOTEBOOK_DIR, "guaiguai_ghibli_film.mp4")
print(f"\n=== 实际视频处理 ===")
print(f"视频文件: {video_path}")
print(f"文件大小: {os.path.getsize(video_path) / 1024:.1f} KB")

messages_video = [{"role": "user", "content": [{"type": "video", "video": video_path}, {"type": "text", "text": "Describe this video."}]}]
prompt_v = processor.apply_chat_template(messages_video, tokenize=False, add_generation_prompt=True)
inputs_v = processor(text=[prompt_v], videos=[video_path], return_tensors="pt")

n_video_tok = (inputs_v["input_ids"] == config.video_token_id).sum().item()
n_img_tok = (inputs_v["input_ids"] == config.image_token_id).sum().item()
total_tok = inputs_v["input_ids"].shape[1]
print(f"\n视频 Token (video_token) 数量: {n_video_tok}")
print(f"图片 Token (image_token) 数量: {n_img_tok}")
print(f"文本 Token 数量: {total_tok - n_video_tok - n_img_tok}")
print(f"总 Token 数: {total_tok}")

if "video_grid_thw" in inputs_v:
    grid = inputs_v["video_grid_thw"][0]
    print(f"\n视频 Grid (T, H, W): {grid.tolist()}")
    print(f"  时间维度: {grid[0].item()} 个时间 Patch")
    print(f"  空间高度: {grid[1].item()} 个空间 Patch")
    print(f"  空间宽度: {grid[2].item()} 个空间 Patch")
    print(f"  视频 Token 数 = T x H x W = {(grid[0] * grid[1] * grid[2]).item()}")

video_tok = tokenizer.decode([config.video_token_id])
print(f"\n视频 Token 标记: {video_tok!r} (ID: {config.video_token_id})")
print(f"图片 Token 标记: {tokenizer.decode([config.image_token_id])!r} (ID: {config.image_token_id})")
print(f"注意: 图片和视频使用不同的占位 Token，模型可以区分它们。")

print(f"\n=== 对比: 同样内容的 Token 数量 ===")
print(f"  乖乖图片 (512x683): 336 个视觉 Token")
print(f"  视频 (328x640, 13秒): {n_video_tok} 个视频 Token")
print(f'  一句话\u201c请描述这张图片\u201d: ~5 个文本 Token')

=== 视频 Token 编码参数 ===
时间 Patch 尺寸: 2 (连续 2 帧合并)
空间合并尺寸: 2x2
Patch 尺寸: 16x16
每个视频 Token 实际编码: 2 帧 x 2x2 空间 Patch = 8 个原始 Patch

=== 实际视频处理 ===
视频文件: /Users/rongtao/Desktop/LLMPrimer/01-token_journey/guaiguai_ghibli_film.mp4
文件大小: 545.2 KB



视频 Token (video_token) 数量: 1664
图片 Token (image_token) 数量: 0
文本 Token 数量: 123
总 Token 数: 1787

视频 Grid (T, H, W): [13, 32, 16]
  时间维度: 13 个时间 Patch
  空间高度: 32 个空间 Patch
  空间宽度: 16 个空间 Patch
  视频 Token 数 = T x H x W = 6656

视频 Token 标记: '<|video_pad|>' (ID: 248057)
图片 Token 标记: '<|image_pad|>' (ID: 248056)
注意: 图片和视频使用不同的占位 Token，模型可以区分它们。

=== 对比: 同样内容的 Token 数量 ===
  乖乖图片 (512x683): 336 个视觉 Token
  视频 (328x640, 13秒): 1664 个视频 Token
  一句话“请描述这张图片”: ~5 个文本 Token


### 视频流式推理（带 KV Cache 加速）

既然 Qwen3.5 也支持视频输入，让我们也试试用开源视频做一次流式推理。视频的 Token 数量更多（1664 个），KV Cache 的加速效果会更明显：

In [3]:
video_path = os.path.join(NOTEBOOK_DIR, "guaiguai_ghibli_film.mp4")
print(f"视频文件: {video_path}")

question = "视频里发生了什么？挑重点，禁止啦嘟 /no_think"
messages_v = [
    {
        "role": "user",
        "content": [
            {"type": "video", "video": video_path},
            {"type": "text", "text": question},
        ],
    }
]

inputs_v = processor.apply_chat_template(
    messages_v,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
)

n_video_tok = (inputs_v["input_ids"] == config.video_token_id).sum().item()
print(f"提问: {question}")
print(f"视频 Token 数量: {n_video_tok}")
print(f"\n流式生成中: ", end="", flush=True)

streamer_v = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

generation_kwargs_v = dict(
    **inputs_v,
    streamer=streamer_v,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    repetition_penalty=1.0,
    do_sample=True,
)

import time
t0_v = time.time()

thread_v = Thread(target=model.generate, kwargs=generation_kwargs_v)
thread_v.start()

for new_text in streamer_v:
    print(new_text, end="", flush=True)

thread_v.join()
elapsed_v = time.time() - t0_v
print(f"\n【结束啦】耗时 {elapsed_v:.1f}s")

视频文件: /Users/rongtao/Desktop/LLMPrimer/01-token_journey/guaiguai_ghibli_film.mp4
提问: 视频里发生了什么？挑重点，禁止啦嘟 /no_think
视频 Token 数量: 1664

流式生成中: 

视频中

展示

了一只

白色的

...

有些

呆

萌

。



【结束啦】耗时 443.1s


## 小结

本节我们深入探索了视频如何变成 Token：

| 步骤 | 操作 | 结果 |
|------|------|------|
| 1 | 采样 (1-2 FPS) | 不是每帧都处理，降低数据量 |
| 2 | 时间 Patch (2帧合并) | 连续2帧合并为1个时间Patch |
| 3 | 空间 Patch (16×16) | 和图片一样切分空间Patch |
| 4 | 空间合并 (2×2) | 相邻Patch合并，减少Token数 |
| 5 | 添加位置编码 | mRoPE编码时间+空间位置 |

核心洞察：**视频 Token = 时间维度 × 空间维度**。每个视频 Token 实际编码了 2×2×2=8 个原始 Patch 的信息。视频比图片多了“时间”这个维度，因此 Token 数量也更多。

对比：
- 乖乖图片 (512×683): 336 个视觉 Token
- 视频 (328×640, 13秒): 1664 个视频 Token
- 一句话: ~5 个文本 Token

👉 下一节：[01-6 声音怎么变成 Token](./01-6-audio_to_token.ipynb)